# Setup

In [1]:
from pprint import pprint
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Seed
seed = 42

# sklearn config
from sklearn import set_config
set_config(transform_output='pandas', display='text')

# 1. Voting

- Classification
  - Hard voting: [0, 0, 1] -> 0
  - Soft voting
    - P(class=0) = [0.2, 0.6, 0.9] -> 0.567
    - P(class=1) = [0.8, 0.4, 0.1] -> 0.433
    - Prediction: class 0
- Regression
  - [3, 6, 9] -> 6

## 1) Classification

> Note) for `VotingClassifier(voting='soft')`, the all estimators must support `.predict_proba()`.

<details>
<summary>Models with <b>.predict_proba()</b></summary>

- `LogisticRegression`
- `KNeighborsClassifier`
- `RadiusNeighborsClassifier`
- `SVC(probability=True)`
- `NuSVC(probability=True)`
- `GaussianNB`
- `MultinomialNB`
- `BernoulliNB`
- `ComplementNB`
- `CategoricalNB`
- `LinearDiscriminantAnalysis`
- `QuadraticDiscriminantAnalysis`
- `GaussianProcessClassifier`
- `MLPClassifier`
- `SGDClassifier(loss="log_loss")`
- `SGDClassifier(loss="modified_huber")`

</details>

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

# Data
X, y = load_iris(
    return_X_y=True,
    as_frame=True,
)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    stratify=y,
    random_state=seed,
)

# Model
base_lr = LogisticRegression()
base_nb = GaussianNB()
base_dt = DecisionTreeClassifier()

model = VotingClassifier(
    estimators=[
        ('lr', base_lr),
        ('nb', base_nb),
        ('dt', base_dt),
    ],
    voting='soft',          # {'hard', 'soft'}.
    weights=[1, 2, 3],      # weights for estimators.
    n_jobs=None,
    verbose=True,
)

# Fit
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

y_pred[:5]

[Voting] ....................... (1 of 3) Processing lr, total=   0.0s
[Voting] ....................... (2 of 3) Processing nb, total=   0.0s
[Voting] ....................... (3 of 3) Processing dt, total=   0.0s


array([0, 2, 1, 1, 0])

## 2) Regression

In [3]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

# Data
X, y = fetch_california_housing(
    return_X_y=True,
    as_frame=True,
)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=seed,
)

# Model
base_lr = LinearRegression()
base_ridge = Ridge()
base_dt = DecisionTreeRegressor()

model = VotingRegressor(
    estimators=[
        ('lr', base_lr),
        ('ridge', base_ridge),
        ('dt', base_dt),
    ],
    weights=[1, 2, 3],      # weights for estimators.
    n_jobs=None,
    verbose=True,
)

# Fit
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

y_pred[:5]

[Voting] ....................... (1 of 3) Processing lr, total=   0.0s
[Voting] .................... (2 of 3) Processing ridge, total=   0.0s
[Voting] ....................... (3 of 3) Processing dt, total=   0.1s


array([0.5666004 , 1.48348657, 3.8546456 , 2.8529797 , 2.43086562])

# 2. Bagging

- Train same models with different subset of data.
  - Bootstrapping: sampling with replacement (duplication O)
- Bagging primarily reduces variance -> tends to help unstable/high-variance estimators more than already-stable estimators.

## 1) Classification

In [4]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression

# Data
X, y = load_iris(
    return_X_y=True,
    as_frame=True,
)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    stratify=y,
    random_state=seed,
)

# Model
base_lr = LogisticRegression()

model = BaggingClassifier(
    estimator=base_lr,              # base model.
    n_estimators=3,                 # number of estimators.
    max_samples=0.8,                # use 80% of samples for each estimator.
    max_features=0.5,               # use 50% of features for each estimator.
    bootstrap=True,                 # if False, sample without replacement.
    bootstrap_features=False,       # if True, a same feature can be sampled multiple times.
    oob_score=True,                 # out-of-bag score, i.e. validate using samples not sampled.
    warm_start=False,               # if True, a second `fit` keeps existing estimators and trains only additional estimators.

    n_jobs=None,
    random_state=seed,
    verbose=2,
)

# Fit
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

print(f"\n- y_pred: {y_pred[:5]}")
print(f"- oob_score (accuracy): {model.oob_score_:.4f}\n")

Building estimator 1 of 3 for this parallel run (total 3)...
Building estimator 2 of 3 for this parallel run (total 3)...
Building estimator 3 of 3 for this parallel run (total 3)...

- y_pred: [0 2 1 1 0]
- oob_score (accuracy): 0.8583



[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\sklearn\ensemble\_bagging.py:917: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\sklearn\ensemble\_bagging.py:923: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


## 2) Regression

In [5]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression

# Data
X, y = fetch_california_housing(
    return_X_y=True,
    as_frame=True,
)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=seed,
)

# Model
base_lr = LinearRegression()

model = BaggingRegressor(
    estimator=base_lr,              # base model.
    n_estimators=3,                 # number of estimators.
    max_samples=0.8,                # use 80% of samples for each estimator.
    max_features=0.5,               # use 50% of features for each estimator.
    bootstrap=True,                 # if False, sample without replacement.
    bootstrap_features=False,       # if True, a same feature can be sampled multiple times.
    oob_score=True,                 # out-of-bag score, i.e. validate using samples not sampled.
    warm_start=False,               # if True, a second `fit` keeps existing estimators and trains only additional estimators.

    n_jobs=None,
    random_state=seed,
    verbose=2,
)

# Fit
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

print(f"\n- y_pred: {y_pred[:5]}")
print(f"- oob_score (R^2): {model.oob_score_:.4f}\n")

Building estimator 1 of 3 for this parallel run (total 3)...
Building estimator 2 of 3 for this parallel run (total 3)...
Building estimator 3 of 3 for this parallel run (total 3)...

- y_pred: [1.2465866  1.68343525 1.91364914 2.74806986 2.32614892]
- oob_score (R^2): -0.1755



[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished
c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\sklearn\ensemble\_bagging.py:1464: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


# 3. Stacking

## 1) Out-of-Fold Stacking

- n_samples = 100
- n_classes = 3
- n_estimators = 4
- K-fold cross-validation
  - `stack_method='predict'`: 4 x 100 predictions
  - `stack_method='predict_proba'`: 4 x 3 x 100 predictions
- Meta-model training
  - `predict`: 4 x 100 predictions -> 100 labels
  - `predict_proba`: 4 x 3 x 100 predictions -> 100 labels

> Note) Below warning is caused by `set_config(transform_output='pandas')` but can be ignored.

```markdown
UserWarning: X has feature names, but LogisticRegression was fitted without feature names
```

In [6]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

# Data
X, y = load_iris(
    return_X_y=True,
    as_frame=True,
)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    stratify=y,
    random_state=seed,
)

# Model
base_lr = LogisticRegression()
base_nb = GaussianNB()
base_dt = DecisionTreeClassifier()
final_lr = LogisticRegression()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=seed,
)

model = StackingClassifier(
    estimators=[
        ('lr', base_lr),
        ('nb', base_nb),
        ('dt', base_dt),
    ],
    final_estimator=final_lr,
    cv=cv,
    stack_method='predict_proba',   # meta features, {'predict', 'predict_proba'}.
    passthrough=False,              # if True, append original features to meta features.
    n_jobs=None,
    verbose=2,
)

# Fit
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

y_pred[:5]

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished
c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\sklearn\utils\validation.py:2823: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


array([0, 2, 1, 1, 0])

## 2) Blending

- Holdout-based stacking.
- n_samples = 100
  - base-training set = 80 -> train base models
  - meta-training set = 20 -> train meta-model
    - aka blending set
- n_classes = 3
- n_estimators = 4
- Meta-model training
  - `stack_method='predict'`: 4 x 20 predictions -> 20 labels
  - `stack_method='predict_proba'`: 4 x 3 x 20 predictions -> 20 labels

In [7]:
from sklearn.datasets import load_iris
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Data
X, y = load_iris(
    return_X_y=True,
    as_frame=True,
)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    stratify=y,
    random_state=seed,
)
X_base, X_blend, y_base, y_blend = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=seed,
)

# Base models
estimators=[
    ('lr', base_lr),
    ('nb', base_nb),
    ('dt', base_dt),
]

for _, estimator in estimators:
    estimator.fit(X_base, y_base)  # Base models never train on X_blend.

# Meta-model
model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv="prefit",  # Reuse the fitted base models; do not refit them.
    stack_method="predict_proba",
    passthrough=False,
)

model.fit(X_blend, y_blend)  # Trains only the meta-model.

# Predict
y_pred = model.predict(X_test)

y_pred[:5]

c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\sklearn\utils\validation.py:2823: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


array([0, 2, 1, 1, 0])